# Crypto Trading Extension for Autonomous Traders

This notebook extends the autonomous traders simulation to support **cryptocurrency trading** alongside equities.

**Additions:**
- **CoinGecko MCP Server** — real-time crypto pricing for 100+ cryptocurrencies
- **Fractional quantities** — crypto holdings support decimal amounts (e.g., 0.5 BTC)
- **24/7 trading** — crypto markets never close
- **Updated trader strategies** — Cathie now trades crypto directly, others can optionally

In [1]:
import os, json, sqlite3, asyncio, secrets, string, random, requests
from contextlib import AsyncExitStack
from datetime import datetime, timezone
from functools import lru_cache
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from polygon import RESTClient
from agents import Agent, Runner, TracingProcessor, trace, add_trace_processor
from agents.mcp import MCPServerStdio
import mcp
from mcp.client.stdio import stdio_client
from mcp import StdioServerParameters

In [2]:
load_dotenv(override=True)

NODE_BIN = "/Users/kelvini/.nvm/versions/node/v20.19.0/bin/node"
UV_BIN = "/Users/kelvini/.local/bin/uv"
UVX_BIN = "/Users/kelvini/.local/bin/uvx"
MCP_REMOTE_JS = "/Users/kelvini/.npm/_npx/705d23756ff7dacc/node_modules/mcp-remote/dist/proxy.js"
BRAVE_SEARCH_JS = "/Users/kelvini/.npm/_npx/be9bcbed6978f068/node_modules/@modelcontextprotocol/server-brave-search/dist/index.js"
MEMORY_LIBSQL_JS = "/Users/kelvini/.npm/_npx/a580a3a6affbe32e/node_modules/mcp-memory-libsql/dist/index.js"
COINGECKO_MCP_URL = "https://mcp.api.coingecko.com/mcp"

polygon_api_key = os.getenv("POLYGON_API_KEY")
polygon_plan = os.getenv("POLYGON_PLAN")
is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

DB = os.path.join(os.getcwd(), "crypto_accounts.db")

## Database
SQLite database for persisting accounts, logs, and market data.

In [3]:
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS accounts (name TEXT PRIMARY KEY, account TEXT)')
    cursor.execute('CREATE TABLE IF NOT EXISTS logs (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT, datetime DATETIME, type TEXT, message TEXT)')
    cursor.execute('CREATE TABLE IF NOT EXISTS market (date TEXT PRIMARY KEY, data TEXT)')
    conn.commit()

def write_account(name, account_dict):
    with sqlite3.connect(DB) as conn:
        conn.execute('INSERT INTO accounts (name, account) VALUES (?, ?) ON CONFLICT(name) DO UPDATE SET account=excluded.account', (name.lower(), json.dumps(account_dict)))
        conn.commit()

def read_account(name):
    with sqlite3.connect(DB) as conn:
        row = conn.execute('SELECT account FROM accounts WHERE name = ?', (name.lower(),)).fetchone()
        return json.loads(row[0]) if row else None

def write_log(name, type, message):
    with sqlite3.connect(DB) as conn:
        conn.execute('INSERT INTO logs (name, datetime, type, message) VALUES (?, datetime("now"), ?, ?)', (name.lower(), type, message))
        conn.commit()

def read_log(name, last_n=10):
    with sqlite3.connect(DB) as conn:
        rows = conn.execute('SELECT datetime, type, message FROM logs WHERE name = ? ORDER BY datetime DESC LIMIT ?', (name.lower(), last_n)).fetchall()
        return reversed(rows)

def write_market(date, data):
    with sqlite3.connect(DB) as conn:
        conn.execute('INSERT INTO market (date, data) VALUES (?, ?) ON CONFLICT(date) DO UPDATE SET data=excluded.data', (date, json.dumps(data)))
        conn.commit()

def read_market(date):
    with sqlite3.connect(DB) as conn:
        row = conn.execute('SELECT data FROM market WHERE date = ?', (date,)).fetchone()
        return json.loads(row[0]) if row else None

print(f"Database initialized: {DB}")

Database initialized: /Users/kelvini/Andela-LLM-Engineering/agents/6_mcp/community_contributions/Wakanda-Kelvin-Isievwore/crypto_accounts.db


## Crypto Price Helper
Maps trading symbols to CoinGecko IDs and provides REST API fallback for portfolio valuation.

In [4]:
CRYPTO_SYMBOLS = {
    "BTC": "bitcoin", "ETH": "ethereum", "SOL": "solana", "DOGE": "dogecoin",
    "ADA": "cardano", "XRP": "ripple", "DOT": "polkadot", "AVAX": "avalanche-2",
    "MATIC": "matic-network", "LINK": "chainlink", "UNI": "uniswap", "ATOM": "cosmos",
}

def is_crypto(symbol):
    return symbol.upper() in CRYPTO_SYMBOLS

def get_crypto_price(symbol):
    coin_id = CRYPTO_SYMBOLS.get(symbol.upper())
    if not coin_id:
        return 0.0
    try:
        resp = requests.get("https://api.coingecko.com/api/v3/simple/price",
                            params={"ids": coin_id, "vs_currencies": "usd"}, timeout=10)
        resp.raise_for_status()
        return resp.json().get(coin_id, {}).get("usd", 0.0)
    except Exception as e:
        print(f"CoinGecko error for {symbol}: {e}")
        return 0.0

print(f"BTC: ${get_crypto_price('BTC'):,.2f}")
print(f"ETH: ${get_crypto_price('ETH'):,.2f}")

BTC: $71,388.00
ETH: $2,174.90


## Market Module
Stock prices via Polygon API + crypto via CoinGecko. `get_share_price()` auto-routes crypto.

In [5]:
def is_market_open():
    try:
        client = RESTClient(polygon_api_key)
        return client.get_market_status().market == "open"
    except:
        return False

@lru_cache(maxsize=2)
def get_market_for_prior_date(today):
    market_data = read_market(today)
    if not market_data:
        client = RESTClient(polygon_api_key)
        probe = client.get_previous_close_agg("SPY")[0]
        last_close = datetime.fromtimestamp(probe.timestamp / 1000, tz=timezone.utc).date()
        results = client.get_grouped_daily_aggs(last_close, adjusted=True, include_otc=False)
        market_data = {r.ticker: r.close for r in results}
        write_market(today, market_data)
    return market_data

def get_share_price(symbol):
    if is_crypto(symbol):
        return get_crypto_price(symbol)
    if polygon_api_key:
        try:
            if is_paid_polygon:
                client = RESTClient(polygon_api_key)
                result = client.get_snapshot_ticker("stocks", symbol)
                return result.min.close or result.prev_day.close
            else:
                today = datetime.now().date().strftime("%Y-%m-%d")
                return get_market_for_prior_date(today).get(symbol, 0.0)
        except Exception as e:
            print(f"Polygon error: {e}")
    return float(random.randint(1, 100))

print(f"get_share_price('BTC') = ${get_share_price('BTC'):,.2f}")
print(f"get_share_price('AAPL') = ${get_share_price('AAPL'):,.2f}")

get_share_price('BTC') = $71,388.00
get_share_price('AAPL') = $251.64


## Account Model
`Account` and `Transaction` with fractional quantity support for crypto. Holdings are `dict[str, float]`.

In [11]:
INITIAL_BALANCE = 10_000.0
SPREAD = 0.002

class Transaction(BaseModel):
    symbol: str
    quantity: float
    price: float
    timestamp: str
    rationale: str
    def total(self): return self.quantity * self.price

class Account(BaseModel):
    name: str
    balance: float
    strategy: str
    holdings: dict[str, float]
    transactions: list[Transaction]
    portfolio_value_time_series: list[tuple[str, float]]

    @classmethod
    def get(cls, name):
        fields = read_account(name.lower())
        if not fields:
            fields = {"name": name.lower(), "balance": INITIAL_BALANCE, "strategy": "",
                      "holdings": {}, "transactions": [], "portfolio_value_time_series": []}
            write_account(name, fields)
        return cls(**fields)

    def save(self):
        write_account(self.name.lower(), self.model_dump())

    def reset(self, strategy):
        self.balance = INITIAL_BALANCE
        self.strategy = strategy
        self.holdings = {}
        self.transactions = []
        self.portfolio_value_time_series = []
        self.save()

    def buy_shares(self, symbol, quantity, rationale):
        price = get_share_price(symbol)
        buy_price = price * (1 + SPREAD)
        total_cost = buy_price * quantity
        if total_cost > self.balance: raise ValueError("Insufficient funds.")
        if price == 0: raise ValueError(f"Unrecognized symbol {symbol}")
        self.holdings[symbol] = self.holdings.get(symbol, 0) + quantity
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.transactions.append(Transaction(symbol=symbol, quantity=quantity, price=buy_price, timestamp=ts, rationale=rationale))
        self.balance -= total_cost
        self.save()
        write_log(self.name, "account", f"Bought {quantity} of {symbol}")
        return "Completed. Latest details:\n" + self.report()

    def sell_shares(self, symbol, quantity, rationale):
        if self.holdings.get(symbol, 0) < quantity: raise ValueError(f"Cannot sell {quantity} of {symbol}.")
        price = get_share_price(symbol)
        sell_price = price * (1 - SPREAD)
        self.holdings[symbol] -= quantity
        if self.holdings[symbol] <= 0.0001: del self.holdings[symbol]
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.transactions.append(Transaction(symbol=symbol, quantity=-quantity, price=sell_price, timestamp=ts, rationale=rationale))
        self.balance += sell_price * quantity
        self.save()
        write_log(self.name, "account", f"Sold {quantity} of {symbol}")
        return "Completed. Latest details:\n" + self.report()

    def calculate_portfolio_value(self):
        return self.balance + sum(get_share_price(s) * q for s, q in self.holdings.items())

    def calculate_profit_loss(self, pv):
        return pv - sum(t.total() for t in self.transactions) - self.balance

    def get_holdings(self): return self.holdings
    def list_transactions(self): return [t.model_dump() for t in self.transactions]

    def report(self):
        pv = self.calculate_portfolio_value()
        self.portfolio_value_time_series.append((datetime.now().strftime("%Y-%m-%d %H:%M:%S"), pv))
        self.save()
        data = self.model_dump()
        data["total_portfolio_value"] = pv
        data["total_profit_loss"] = self.calculate_profit_loss(pv)
        write_log(self.name, "account", "Retrieved account details")
        return json.dumps(data)

    def get_strategy(self):
        write_log(self.name, "account", "Retrieved strategy")
        return self.strategy

    def change_strategy(self, strategy):
        self.strategy = strategy
        self.save()
        write_log(self.name, "account", "Changed strategy")
        return "Changed strategy"

print("Account model ready (fractional crypto support)")

Account model ready (fractional crypto support)


## MCP Servers
We write crypto-adapted MCP server `.py` files to the notebook directory using `%%writefile`.
These run as subprocesses via `uv run`.

In [12]:
%%writefile crypto_accounts_server.py
import os, json, sqlite3, random, requests
from datetime import datetime, timezone
from pydantic import BaseModel
from mcp.server.fastmcp import FastMCP
from dotenv import load_dotenv

load_dotenv(override=True)
DB = os.path.join(os.path.dirname(os.path.abspath(__file__)), "crypto_accounts.db")
polygon_api_key = os.getenv("POLYGON_API_KEY")
is_paid_polygon = os.getenv("POLYGON_PLAN") == "paid"

def write_account(name, d):
    with sqlite3.connect(DB) as c: c.execute('INSERT INTO accounts (name, account) VALUES (?,?) ON CONFLICT(name) DO UPDATE SET account=excluded.account', (name.lower(), json.dumps(d))); c.commit()
def read_account(name):
    with sqlite3.connect(DB) as c:
        r = c.execute('SELECT account FROM accounts WHERE name=?', (name.lower(),)).fetchone()
        return json.loads(r[0]) if r else None
def write_log(name, tp, msg):
    with sqlite3.connect(DB) as c: c.execute('INSERT INTO logs (name,datetime,type,message) VALUES (?,datetime("now"),?,?)', (name.lower(),tp,msg)); c.commit()

CRYPTO = {"BTC":"bitcoin","ETH":"ethereum","SOL":"solana","DOGE":"dogecoin","ADA":"cardano","XRP":"ripple","DOT":"polkadot","AVAX":"avalanche-2","MATIC":"matic-network","LINK":"chainlink","UNI":"uniswap","ATOM":"cosmos"}
def is_crypto(s): return s.upper() in CRYPTO
def get_crypto_price(s):
    cid = CRYPTO.get(s.upper())
    if not cid: return 0.0
    try:
        r = requests.get("https://api.coingecko.com/api/v3/simple/price", params={"ids":cid,"vs_currencies":"usd"}, timeout=10); r.raise_for_status()
        return r.json().get(cid,{}).get("usd",0.0)
    except: return 0.0

def get_share_price(symbol):
    if is_crypto(symbol): return get_crypto_price(symbol)
    if polygon_api_key:
        try:
            from polygon import RESTClient
            client = RESTClient(polygon_api_key)
            if is_paid_polygon:
                result = client.get_snapshot_ticker("stocks", symbol)
                return result.min.close or result.prev_day.close
            probe = client.get_previous_close_agg("SPY")[0]
            last_close = datetime.fromtimestamp(probe.timestamp/1000, tz=timezone.utc).date()
            prices = {r.ticker: r.close for r in client.get_grouped_daily_aggs(last_close, adjusted=True, include_otc=False)}
            return prices.get(symbol, 0.0)
        except Exception as e: print(f"Polygon error: {e}")
    return float(random.randint(1,100))

SPREAD = 0.002
INITIAL_BALANCE = 10_000.0

class Transaction(BaseModel):
    symbol:str; quantity:float; price:float; timestamp:str; rationale:str
    def total(self): return self.quantity*self.price

class Account(BaseModel):
    name:str; balance:float; strategy:str; holdings:dict[str,float]
    transactions:list[Transaction]; portfolio_value_time_series:list[tuple[str,float]]
    @classmethod
    def get(cls, name):
        f = read_account(name.lower())
        if not f: f={"name":name.lower(),"balance":INITIAL_BALANCE,"strategy":"","holdings":{},"transactions":[],"portfolio_value_time_series":[]}; write_account(name,f)
        return cls(**f)
    def save(self): write_account(self.name.lower(), self.model_dump())
    def buy_shares(self, symbol, quantity, rationale):
        p=get_share_price(symbol); bp=p*(1+SPREAD); tc=bp*quantity
        if tc>self.balance: raise ValueError("Insufficient funds.")
        if p==0: raise ValueError(f"Unrecognized symbol {symbol}")
        self.holdings[symbol]=self.holdings.get(symbol,0)+quantity
        ts=datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.transactions.append(Transaction(symbol=symbol,quantity=quantity,price=bp,timestamp=ts,rationale=rationale))
        self.balance-=tc; self.save(); write_log(self.name,"account",f"Bought {quantity} of {symbol}")
        return "Completed. Latest details:\n"+self.report()
    def sell_shares(self, symbol, quantity, rationale):
        if self.holdings.get(symbol,0)<quantity: raise ValueError(f"Cannot sell {quantity} of {symbol}.")
        p=get_share_price(symbol); sp=p*(1-SPREAD)
        self.holdings[symbol]-=quantity
        if self.holdings[symbol]<=0.0001: del self.holdings[symbol]
        ts=datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.transactions.append(Transaction(symbol=symbol,quantity=-quantity,price=sp,timestamp=ts,rationale=rationale))
        self.balance+=sp*quantity; self.save(); write_log(self.name,"account",f"Sold {quantity} of {symbol}")
        return "Completed. Latest details:\n"+self.report()
    def calculate_portfolio_value(self): return self.balance+sum(get_share_price(s)*q for s,q in self.holdings.items())
    def calculate_profit_loss(self, pv): return pv-sum(t.total() for t in self.transactions)-self.balance
    def report(self):
        pv=self.calculate_portfolio_value()
        self.portfolio_value_time_series.append((datetime.now().strftime("%Y-%m-%d %H:%M:%S"),pv)); self.save()
        d=self.model_dump(); d["total_portfolio_value"]=pv; d["total_profit_loss"]=self.calculate_profit_loss(pv)
        return json.dumps(d)
    def get_strategy(self): return self.strategy
    def change_strategy(self, strategy): self.strategy=strategy; self.save(); return "Changed strategy"

mcp_server = FastMCP("crypto_accounts_server")

@mcp_server.tool()
async def get_balance(name: str) -> float:
    """Get cash balance. Args: name: account holder name"""
    return Account.get(name).balance

@mcp_server.tool()
async def get_holdings(name: str) -> dict[str, float]:
    """Get holdings (stocks+crypto). Args: name: account holder name"""
    return Account.get(name).holdings

@mcp_server.tool()
async def buy_shares(name: str, symbol: str, quantity: float, rationale: str) -> str:
    """Buy stock or crypto (fractional OK). Args: name: account name, symbol: e.g. AAPL or BTC, quantity: amount, rationale: why"""
    return Account.get(name).buy_shares(symbol, quantity, rationale)

@mcp_server.tool()
async def sell_shares(name: str, symbol: str, quantity: float, rationale: str) -> str:
    """Sell stock or crypto (fractional OK). Args: name: account name, symbol: ticker, quantity: amount, rationale: why"""
    return Account.get(name).sell_shares(symbol, quantity, rationale)

@mcp_server.tool()
async def change_strategy(name: str, strategy: str) -> str:
    """Change investment strategy. Args: name: account name, strategy: new strategy"""
    return Account.get(name).change_strategy(strategy)

@mcp_server.resource("accounts://crypto_accounts_server/{name}")
async def read_account_resource(name: str) -> str:
    return Account.get(name.lower()).report()

@mcp_server.resource("accounts://strategy/{name}")
async def read_strategy_resource(name: str) -> str:
    return Account.get(name.lower()).get_strategy()

if __name__ == "__main__":
    mcp_server.run(transport='stdio')

Writing crypto_accounts_server.py


In [13]:
%%writefile crypto_market_server.py
import os, random, requests
from datetime import datetime, timezone
from mcp.server.fastmcp import FastMCP
from dotenv import load_dotenv

load_dotenv(override=True)
polygon_api_key = os.getenv("POLYGON_API_KEY")
is_paid_polygon = os.getenv("POLYGON_PLAN") == "paid"

CRYPTO = {"BTC":"bitcoin","ETH":"ethereum","SOL":"solana","DOGE":"dogecoin","ADA":"cardano","XRP":"ripple","DOT":"polkadot","AVAX":"avalanche-2","MATIC":"matic-network","LINK":"chainlink","UNI":"uniswap","ATOM":"cosmos"}
def is_crypto(s): return s.upper() in CRYPTO
def get_crypto_price(s):
    cid = CRYPTO.get(s.upper())
    if not cid: return 0.0
    try:
        r = requests.get("https://api.coingecko.com/api/v3/simple/price", params={"ids":cid,"vs_currencies":"usd"}, timeout=10); r.raise_for_status()
        return r.json().get(cid,{}).get("usd",0.0)
    except: return 0.0

def get_share_price(symbol):
    if is_crypto(symbol): return get_crypto_price(symbol)
    if polygon_api_key:
        try:
            from polygon import RESTClient
            client = RESTClient(polygon_api_key)
            if is_paid_polygon:
                result = client.get_snapshot_ticker("stocks", symbol)
                return result.min.close or result.prev_day.close
            probe = client.get_previous_close_agg("SPY")[0]
            last_close = datetime.fromtimestamp(probe.timestamp/1000, tz=timezone.utc).date()
            prices = {r.ticker: r.close for r in client.get_grouped_daily_aggs(last_close, adjusted=True, include_otc=False)}
            return prices.get(symbol, 0.0)
        except Exception as e: print(f"Polygon error: {e}")
    return float(random.randint(1,100))

mcp = FastMCP("crypto_market_server")

@mcp.tool()
async def lookup_share_price(symbol: str) -> float:
    """Get current price of a stock or crypto symbol. Args: symbol: e.g. AAPL, BTC, ETH"""
    return get_share_price(symbol)

@mcp.tool()
async def list_crypto_symbols() -> dict:
    """List all available cryptocurrency symbols."""
    return dict(CRYPTO)

if __name__ == "__main__":
    mcp.run(transport='stdio')

Writing crypto_market_server.py


In [14]:
%%writefile crypto_push_server.py
import os, requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from mcp.server.fastmcp import FastMCP

load_dotenv(override=True)
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")

mcp = FastMCP("crypto_push_server")

class PushModelArgs(BaseModel):
    message: str = Field(description="A brief message to push")

@mcp.tool()
def push(args: PushModelArgs):
    """Send a push notification with this brief message"""
    print(f"Push: {args.message}")
    requests.post("https://api.pushover.net/1/messages.json",
                  data={"user": pushover_user, "token": pushover_token, "message": args.message})
    return "Push notification sent"

if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing crypto_push_server.py


## CoinGecko MCP Server Test
Verify the CoinGecko MCP remote server connects and list its available tools.

In [15]:
coingecko_params = {
    "command": NODE_BIN,
    "args": [MCP_REMOTE_JS, COINGECKO_MCP_URL],
    "env": {**os.environ},
}

async with MCPServerStdio(params=coingecko_params, client_session_timeout_seconds=60) as server:
    cg_tools = await server.list_tools()

print(f"CoinGecko MCP: {len(cg_tools)} tools available")
for tool in cg_tools:
    print(f"  - {tool.name}: {tool.description[:80]}")

CoinGecko MCP: 50 tools available
  - get_asset_platforms: When using this tool, always use the `jq_filter` parameter to reduce the respons
  - get_id_coins: This endpoint allows you to **query all the metadata (image, websites, socials, 
  - get_list_coins_categories: When using this tool, always use the `jq_filter` parameter to reduce the respons
  - get_new_coins_list: When using this tool, always use the `jq_filter` parameter to reduce the respons
  - get_coins_markets: When using this tool, always use the `jq_filter` parameter to reduce the respons
  - get_coins_top_gainers_losers: When using this tool, always use the `jq_filter` parameter to reduce the respons
  - get_coins_contract: This endpoint allows you to **query all the metadata (image, websites, socials, 
  - get_range_contract_coins_market_chart: When using this tool, always use the `jq_filter` parameter to reduce the respons
  - get_coins_history: When using this tool, always use the `jq_filter` parameter to reduce the 

## MCP Server Params
Configure MCP server connections for traders and researchers.

In [16]:
crypto_trader_mcp_server_params = [
    {"command": UV_BIN, "args": ["run", "crypto_accounts_server.py"]},
    {"command": UV_BIN, "args": ["run", "crypto_push_server.py"]},
    {"command": UV_BIN, "args": ["run", "crypto_market_server.py"]},
    coingecko_params,
]

def crypto_researcher_mcp_server_params(name):
    return [
        {"command": UVX_BIN, "args": ["mcp-server-fetch"]},
        {"command": NODE_BIN, "args": [BRAVE_SEARCH_JS], "env": {**os.environ, "BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}},
        {"command": NODE_BIN, "args": [MEMORY_LIBSQL_JS], "env": {**os.environ, "LIBSQL_URL": f"file:./memory/{name}.db"}},
    ]

print(f"Trader servers: {len(crypto_trader_mcp_server_params)}, Researcher servers: {len(crypto_researcher_mcp_server_params('test'))}")

Trader servers: 4, Researcher servers: 3


## Tracing
Log tracer for tracking agent execution.

In [17]:
ALPHANUM = string.ascii_lowercase + string.digits

def make_trace_id(tag):
    tag += "0"
    return f"trace_{tag}{''.join(secrets.choice(ALPHANUM) for _ in range(32-len(tag)))}"

class LogTracer(TracingProcessor):
    def get_name(self, t):
        name = t.trace_id.split("_")[1]
        return name.split("0")[0] if '0' in name else None

    def on_trace_start(self, t):
        name = self.get_name(t)
        if name: write_log(name, "trace", f"Started: {t.name}")

    def on_trace_end(self, t):
        name = self.get_name(t)
        if name: write_log(name, "trace", f"Ended: {t.name}")

    def _span_msg(self, span, prefix):
        name = self.get_name(span)
        if not name: return
        msg = prefix
        if span.span_data:
            if span.span_data.type: msg += f" {span.span_data.type}"
            if hasattr(span.span_data, "name") and span.span_data.name: msg += f" {span.span_data.name}"
            if hasattr(span.span_data, "server") and span.span_data.server: msg += f" {span.span_data.server}"
        if span.error: msg += f" {span.error}"
        write_log(name, span.span_data.type if span.span_data else "span", msg)

    def on_span_start(self, span): self._span_msg(span, "Started")
    def on_span_end(self, span): self._span_msg(span, "Ended")
    def force_flush(self): pass
    def shutdown(self): pass

print("Tracing ready")

Tracing ready


## Trader Strategies
Updated with crypto trading capabilities for all 4 traders.

In [18]:
CRYPTO_AVAIL = "Available crypto: BTC, ETH, SOL, DOGE, ADA, XRP, DOT, AVAX, MATIC, LINK, UNI, ATOM. You can buy fractional amounts."

warren_strategy = f"""You are Warren, a value-oriented investor (homage to Buffett). You identify high-quality companies below intrinsic value. You may allocate a small portion to BTC as digital gold. {CRYPTO_AVAIL}"""
george_strategy = f"""You are George, an aggressive macro trader (homage to Soros). You exploit mispricings from economic/geopolitical events. You trade equities AND crypto. {CRYPTO_AVAIL}"""
ray_strategy = f"""You are Ray, a systematic risk-parity investor (homage to Dalio). You diversify broadly. Crypto is part of your universe — BTC and ETH as uncorrelated assets. {CRYPTO_AVAIL}"""
cathie_strategy = f"""You are Cathie, an aggressive crypto trader (homage to Cathie Wood). You trade BTC, ETH, SOL, and emerging altcoins directly. Focus the majority of your portfolio on crypto. {CRYPTO_AVAIL}"""

names = ["Warren", "George", "Ray", "Cathie"]
lastnames = ["Patience", "Bold", "Systematic", "Crypto"]
all_strategies = [warren_strategy, george_strategy, ray_strategy, cathie_strategy]

for name, strategy in zip(names, all_strategies):
    Account.get(name).reset(strategy)
print("All traders reset with crypto-enabled strategies")

All traders reset with crypto-enabled strategies


## Prompt Templates
Agent prompts updated with crypto trading capabilities.

In [19]:
if is_realtime_polygon:
    stock_note = "Use get_last_trade for realtime prices."
elif is_paid_polygon:
    stock_note = "Use get_snapshot_ticker for 15-min delayed prices."
else:
    stock_note = "Use lookup_share_price for end-of-day prices."

CRYPTO_NOTE = f"""You have CoinGecko tools for real-time crypto data. {CRYPTO_AVAIL}
Use list_crypto_symbols to see all available crypto."""

def researcher_instructions():
    return f"""You are a financial researcher. Search the web for news and trading opportunities in stocks AND crypto.
Use knowledge graph tools to store/recall info. Current datetime: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"""

def researcher_tool_desc():
    return "Researches online for news and opportunities in stocks and crypto."

def trader_instructions(name):
    return f"""You are {name}, a trader on stock and crypto markets. Account name: {name}.
{stock_note} {CRYPTO_NOTE}
Use tools to research, decide, execute trades. After trading, send a push notification summary, then reply with a 2-3 sentence appraisal."""

def trade_message(name, strategy, account):
    return f"""Look for new opportunities. Use the Researcher tool for news.
{stock_note} {CRYPTO_NOTE}
Trade equities AND/OR crypto. Strategy:\n{strategy}\nAccount:\n{account}\nDatetime: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\nAccount name: {name}. Execute trades, send push summary, reply with appraisal."""

def rebalance_message(name, strategy, account):
    return f"""Examine your portfolio and rebalance if needed. Use the Researcher for news on existing holdings.
{stock_note} {CRYPTO_NOTE}
Strategy:\n{strategy}\nAccount:\n{account}\nDatetime: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\nAccount name: {name}. Execute trades, send push summary, reply with appraisal."""

print("Prompt templates defined")

Prompt templates defined


## Accounts Client
MCP client functions for reading account data from the accounts server.

In [20]:
acct_params = StdioServerParameters(command=UV_BIN, args=["run", "crypto_accounts_server.py"], env=None)

async def read_accounts_resource(name):
    async with stdio_client(acct_params) as streams:
        async with mcp.ClientSession(*streams) as session:
            await session.initialize()
            result = await session.read_resource(f"accounts://crypto_accounts_server/{name}")
            return result.contents[0].text

async def read_strategy_resource(name):
    async with stdio_client(acct_params) as streams:
        async with mcp.ClientSession(*streams) as session:
            await session.initialize()
            result = await session.read_resource(f"accounts://strategy/{name}")
            return result.contents[0].text

print("Accounts client ready")

Accounts client ready


## CryptoTrader Agent
The main trader agent class with MCP servers for accounts, market, crypto, and research.

In [21]:
MAX_TURNS = 30

class CryptoTrader:
    def __init__(self, name, lastname="Trader", model_name="gpt-4o-mini"):
        self.name = name
        self.lastname = lastname
        self.model_name = model_name
        self.do_trade = True

    async def create_agent(self, trader_servers, researcher_servers):
        researcher = Agent(name="Researcher", instructions=researcher_instructions(), model=self.model_name, mcp_servers=researcher_servers)
        tool = researcher.as_tool(tool_name="Researcher", tool_description=researcher_tool_desc())
        return Agent(name=self.name, instructions=trader_instructions(self.name), model=self.model_name, tools=[tool], mcp_servers=trader_servers)

    async def get_account_report(self):
        account = await read_accounts_resource(self.name)
        d = json.loads(account)
        d.pop("portfolio_value_time_series", None)
        return json.dumps(d)

    async def run_agent(self, trader_servers, researcher_servers):
        agent = await self.create_agent(trader_servers, researcher_servers)
        account = await self.get_account_report()
        strategy = await read_strategy_resource(self.name)
        msg = trade_message(self.name, strategy, account) if self.do_trade else rebalance_message(self.name, strategy, account)
        return await Runner.run(agent, msg, max_turns=MAX_TURNS)

    async def run_with_mcp_servers(self):
        async with AsyncExitStack() as s1:
            trader_servers = [await s1.enter_async_context(MCPServerStdio(p, client_session_timeout_seconds=120)) for p in crypto_trader_mcp_server_params]
            async with AsyncExitStack() as s2:
                researcher_servers = [await s2.enter_async_context(MCPServerStdio(p, client_session_timeout_seconds=120)) for p in crypto_researcher_mcp_server_params(self.name)]
                return await self.run_agent(trader_servers, researcher_servers)

    async def run(self):
        try:
            tid = make_trace_id(self.name.lower())
            tname = f"{self.name}-crypto-{'trading' if self.do_trade else 'rebalancing'}"
            with trace(tname, trace_id=tid):
                return await self.run_with_mcp_servers()
        except Exception as e:
            print(f"Error: {self.name}: {e}")
            return None
        finally:
            self.do_trade = not self.do_trade

print("CryptoTrader ready")

CryptoTrader ready


## Crypto Trading Floor
Run all 4 traders concurrently.

In [24]:
traders = [CryptoTrader(n, l, "gpt-4o-mini") for n, l in zip(names, lastnames)]
print("Running all 4 crypto-enabled traders...")
results = await asyncio.gather(*[t.run() for t in traders])

print("\n" + "="*60 + "\nTRADING SUMMARY\n" + "="*60)
for name in names:
    a = Account.get(name)
    pv = a.calculate_portfolio_value()
    pnl = a.calculate_profit_loss(pv)
    print(f"\n{name}: Balance=${a.balance:,.2f} | Holdings={dict(a.holdings)} | Portfolio=${pv:,.2f} ({'+' if pnl>=0 else ''}{pnl:,.2f}) | Trades={len(a.transactions)}")

Running all 4 crypto-enabled traders...


Error initializing MCP server: Connection closed


Error: Cathie: Connection closed


Error cleaning up server: Attempted to exit a cancel scope that isn't the current tasks's current cancel scope
Error cleaning up server: Attempted to exit a cancel scope that isn't the current tasks's current cancel scope
Error cleaning up server: Attempted to exit a cancel scope that isn't the current tasks's current cancel scope
Error cleaning up server: Attempted to exit a cancel scope that isn't the current tasks's current cancel scope
Error cleaning up server: Attempted to exit a cancel scope that isn't the current tasks's current cancel scope
Error cleaning up server: Attempted to exit a cancel scope that isn't the current tasks's current cancel scope
Error cleaning up server: Attempted to exit a cancel scope that isn't the current tasks's current cancel scope


CancelledError: 

Error cleaning up server: Attempted to exit a cancel scope that isn't the current tasks's current cancel scope


Error cleaning up server: Attempted to exit a cancel scope that isn't the current tasks's current cancel scope


## Gradio Dashboard
Live dashboard with stocks + crypto holdings.

In [25]:
import gradio as gr
import pandas as pd
import plotly.express as px

class TraderView:
    def __init__(self, name, lastname, model):
        self.name, self.lastname, self.model = name, lastname, model

    def title(self):
        return f"<div style='text-align:center;font-size:34px;'>{self.name}<span style='color:#ccc;font-size:24px;'> ({self.model}) - {self.lastname}</span></div>"

    def chart(self):
        a = Account.get(self.name)
        df = pd.DataFrame(a.portfolio_value_time_series, columns=["datetime", "value"])
        fig = px.line(df, x="datetime", y="value") if not df.empty else px.line(title="No data")
        fig.update_layout(height=300, margin=dict(l=40,r=20,t=20,b=40), xaxis_title=None, yaxis_title=None)
        return fig

    def holdings(self):
        a = Account.get(self.name)
        h = a.get_holdings()
        if not h: return pd.DataFrame(columns=["Symbol","Qty","Type"])
        return pd.DataFrame([{"Symbol":s,"Qty":round(q,6),"Type":"Crypto" if is_crypto(s) else "Stock"} for s,q in h.items()])

    def txns(self):
        a = Account.get(self.name)
        t = a.list_transactions()
        return pd.DataFrame(t) if t else pd.DataFrame(columns=["timestamp","symbol","quantity","price","rationale"])

    def pv_html(self):
        a = Account.get(self.name)
        pv = a.calculate_portfolio_value()
        pnl = a.calculate_profit_loss(pv)
        c = "green" if pnl>=0 else "red"
        return f"<div style='text-align:center;background-color:{c};'><span style='font-size:32px'>${pv:,.0f}</span> <span style='font-size:24px'>{'+' if pnl>=0 else ''}{pnl:,.0f}</span></div>"

views = [TraderView(n,l,"gpt-4o-mini") for n,l in zip(names, lastnames)]

with gr.Blocks(title="Crypto Traders", theme=gr.themes.Default(primary_hue="emerald")) as ui:
    gr.Markdown("## Autonomous Crypto-Enabled Traders Dashboard")
    with gr.Row():
        for v in views:
            with gr.Column():
                gr.HTML(v.title())
                gr.HTML(v.pv_html)
                gr.Plot(v.chart, show_label=False)
                gr.Dataframe(value=v.holdings, label="Holdings", row_count=(5,"dynamic"), max_height=300)
                gr.Dataframe(value=v.txns, label="Transactions", row_count=(5,"dynamic"), max_height=300)

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.
